# Legacy: single-split applicationThe original `Application_v1.ipynb` fitted RNN-AGT on one fixed 70/30 split andcompared it against Cox models scored in sample. Both halves of that comparisonhave since been replaced, so the old code is not reproduced here.Use **`application/real_data_analysis.ipynb`** for the current analysis.This notebook is retained for one purpose: showing how far the single-splitresult moves under resampling. Reviewer 2 asked why a single split was used;the honest answer is that there was no methodological reason beyond continuitywith the exploratory analysis. This quantifies what that choice cost.

In [ ]:
import os, syssys.path.insert(0, os.path.abspath(".."))   # repository root, so `rnn_agt` importsimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport rnn_agtfrom rnn_agt import data as Dfrom rnn_agt.seeds import make_seedsfrom rnn_agt.train import TrainConfig, train_model, predictfrom rnn_agt.metrics import evaluateprint("rnn_agt", rnn_agt.__version__)

In [ ]:
sys.path.insert(0, os.path.abspath("../experiments"))from run_ablation_and_splits import prepare_real_datafrom rnn_agt.evaluation import run_repeated_splits, summarise, stratified_splitDATA = {"cgd": "../data/cgd.csv",        "crc": "../data/crc.csv"}SEED, N_SPLITS = 20260903, 200cfg = {"rnn_agt": TrainConfig(model="rnn_agt", lr=3e-4, epochs=10,                              pair_sample_s=30, hidden_dim=64, gru_layers=2)}for ds, path in DATA.items():    subs, p = prepare_real_data(path)    outcomes = run_repeated_splits(subs, p, cfg, SEED,                                   n_splits=N_SPLITS, test_frac=0.30)    vals = np.array([o.metrics["rnn_agt"]["test_cindex"] for o in outcomes])    st = summarise(outcomes, "test_cindex")["rnn_agt"]    fig, ax = plt.subplots(figsize=(6, 3.5))    ax.hist(vals, bins=25, color="steelblue", alpha=.75)    ax.axvline(st["mean"], c="firebrick", lw=2, label=f"mean {st['mean']:.3f}")    ax.axvline(st["lo"], c="grey", ls="--", lw=1)    ax.axvline(st["hi"], c="grey", ls="--", lw=1, label="95% range")    ax.set_xlabel("test IPCW C-index"); ax.set_ylabel("splits")    ax.set_title(f"{ds}: RNN-AGT across {N_SPLITS} random splits")    ax.legend(); fig.tight_layout(); plt.show()    print(f"{ds}: mean {st['mean']:.3f}, sd {st['sd']:.3f}, "          f"95% range [{st['lo']:.3f}, {st['hi']:.3f}], "          f"width {st['hi'] - st['lo']:.3f}")

### InterpretationCompare the width of the 95% range against the 0.02 concordance-unit marginthat the original single-split analysis reported on CRC. If the range is widerthan the margin — which is likely at n=128 and n=403 — then a single splitcould plausibly have produced any ordering of the methods, and no conclusionabout superiority follows from it.That is the argument Section 5.6 makes, and this plot is the evidence for it.